# Train and select the SEAD pilot without evaluating outer engines.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT
TORCH_PYTHON = RELEASE_TORCH_PYTHON
CLASSICAL_PYTHON = RELEASE_CLASSICAL_PYTHON


In [ ]:
PLAN = load_json('assets/15B/02_plan.json')

SCRIPTS = {
    'source_check.py': load_text('assets/15B/workers/source_check.py'),
    'prepare_data.py': load_text('assets/15B/workers/prepare_data.py'),
    'train_sead.py': load_text('assets/15B/workers/train_sead.py'),
    'detector_core.py': load_text('assets/15B/workers/detector_core.py'),
    'select_detectors.py': load_text('assets/15B/workers/select_detectors.py')
}


In [ ]:
import hashlib,json,os,platform,subprocess,sys,time,traceback
from pathlib import Path


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()


def canonical(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()).hexdigest()


def put(path,obj):
    temp=path.with_suffix('.tmp');temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(path)


def environment(executable,expected):
    probe="""import json,platform,importlib.metadata as im
v={'python':platform.python_version()}
for k in ['torch','numpy','pandas','scikit-learn','scipy','joblib']:
 try:v[k]=im.version(k)
 except im.PackageNotFoundError:v[k]=None
print(json.dumps(v))
"""
    p=subprocess.run([str(executable),'-c',probe],capture_output=True,text=True,timeout=30)
    if p.returncode:raise RuntimeError('Python metadata hatası: '+p.stderr[-2000:])
    versions=json.loads(p.stdout.strip().splitlines()[-1])
    differences={k:{'expected':v,'actual':versions.get(k)} for k,v in expected.items() if versions.get(k)!=v}
    if differences:raise RuntimeError('Çalışma ortamı değişmiş: '+json.dumps(differences,ensure_ascii=False))
    return dict(executable=str(executable),versions=versions)


def run_stage(executable,script,out,project,timeout):
    log_path=out/(script.stem+'.log')
    env=os.environ.copy();env.update(OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',PYTHONHASHSEED='42')
    with log_path.open('w',encoding='utf-8') as writer,log_path.open('r',encoding='utf-8',errors='replace') as reader:
        child=subprocess.Popen([str(executable),'-u','-X','faulthandler',str(script),str(out),str(project)],
                               stdout=writer,stderr=subprocess.STDOUT,env=env)
        started=time.monotonic()
        try:
            while child.poll() is None:
                chunk=reader.read()
                if chunk:print(chunk,end='',flush=True)
                if time.monotonic()-started>timeout:raise TimeoutError('Aşama süre sınırını aştı: '+script.name)
                time.sleep(2)
            remaining=reader.read()
            if remaining:print(remaining,end='',flush=True)
            if child.returncode:raise RuntimeError(script.name+' çıkış kodu '+str(child.returncode)+'\n'+log_path.read_text(errors='replace')[-6000:])
        except BaseException:
            if child.poll() is None:
                child.terminate()
                try:child.wait(timeout=5)
                except subprocess.TimeoutExpired:child.kill();child.wait()
            raise


def run_pilot(project,torch_python,classical_python,plan,scripts):
    project=Path(project).expanduser().absolute()
    if not project.is_dir():raise FileNotFoundError('PROJECT_DIR bulunamadı: '+str(project))
    plan=dict(plan)
    plan['implementation_sha256']={name:hashlib.sha256(code.encode()).hexdigest() for name,code in scripts.items()}
    fingerprint=canonical(plan)
    out=project/'revision_outputs/15B_FD001_SEAD_pilot'/fingerprint[:20]
    out.mkdir(parents=True,exist_ok=True)
    summary=out/'sead_pilot_summary.json'
    status=dict(status='RUNNING',stage='15B_training_and_selection_only',plan_sha256=fingerprint,
                plan=plan,outer_evaluation_performed=False,original_detector_count_preserved=5)
    put(summary,status)
    try:
        if (out/'plan.json').exists() and json.loads((out/'plan.json').read_text())!=plan:
            raise ValueError('Çalışma klasöründeki plan değişmiş.')
        put(out/'plan.json',plan)
        for name,code in scripts.items():
            path=out/name
            if path.exists() and path.read_text()!=code:raise ValueError('Kaydedilmiş kod değişmiş: '+name)
            path.write_text(code)
        print('ÇIKTI KLASÖRÜ:',out,flush=True)
        status['environments']={
            'torch':environment(torch_python,plan['torch_versions']),
            'classical':environment(classical_python,plan['classical_versions'])}

        for relative,expected in plan['required_hashes'].items():
            source=project/relative
            if not source.is_file() or digest(source)!=expected:
                raise ValueError('Kaynak eksik/değişmiş: '+str(source))
        put(summary,status)
        stages=[('prepare_data',classical_python,['prepared_data.npz','prepare_summary.json']),
                ('train_sead',torch_python,['training_summary.json','trained_outputs.npz','training_history.json',
                                           'sclm_best.pt','sead_raw_error_best.pt','sead_unit_direction_control_best.pt']),
                ('select_detectors',classical_python,['selection_summary.json','selection_scores.npz','detector_trials.json',
                                                     'sclm_latent_scaler.joblib']+
                     ['detector_'+x.replace('-','_')+'.joblib' for x in ['DBSCAN','IF','LOF','OC-SVM','kNN']])]
        for name,executable,files in stages:
            marker=out/(name+'_completed.json')
            stage_environment=status['environments']['torch' if name=='train_sead' else 'classical']
            if marker.is_file():
                saved=json.loads(marker.read_text())
                if saved.get('environment')!=stage_environment:
                    raise ValueError('Tamamlanmış aşamanın Python/paket ortamı değişmiş: '+name)
                if saved['plan_sha256']!=fingerprint or any(not (out/f).is_file() or digest(out/f)!=h for f,h in saved['outputs'].items()):
                    raise ValueError('Tamamlanmış aşamanın kayıtları değişmiş: '+name)
                if set(saved['outputs'])!=set(files):raise ValueError('Aşama dosya kapsamı farklı: '+name)
                print('DOĞRULANMIŞ AŞAMA KULLANILIYOR:',name,flush=True)
                continue
            print('ÇALIŞIYOR:',name,flush=True)
            run_stage(executable,out/(name+'.py'),out,project,plan['stage_timeout_seconds'])
            if not all((out/f).is_file() for f in files):raise ValueError('Aşama dosyaları eksik: '+name)
            put(marker,{'plan_sha256':fingerprint,'environment':stage_environment,'outputs':{f:digest(out/f) for f in files}})
        status['preparation']=json.loads((out/'prepare_summary.json').read_text())
        status['training']=json.loads((out/'training_summary.json').read_text())
        status['selection']=json.loads((out/'selection_summary.json').read_text())
        ok=status['preparation']['status']=='PASS_PREPARE' and status['training']['status']=='PASS_TRAINING' and status['selection']['status']=='PASS_SELECTION'
        if len(status['selection']['selected_detectors'])!=5:raise ValueError('Beş detektör kapsamı eksik.')
        status['status']='PASS_PILOT' if ok else 'REVIEW_REQUIRED'
        status['interpretation']='Training/selection pilot only; no held-out performance claim. Fixed source-loss control is not another classical detector.'
        status['artifact_sha256']={p.name:digest(p) for p in sorted(out.iterdir()) if p.is_file() and p!=summary and p.suffix not in ('.log','.tmp')}
    except (Exception,KeyboardInterrupt):
        status['status']='REVIEW_REQUIRED';status['error']=traceback.format_exc()
    put(summary,status)
    print('\nDURUM:',status['status'])
    if 'error' in status:print(status['error'])
    print('GÖNDERİLECEK DOSYA:',summary)
    return status


In [ ]:
pilot_result = run_pilot(PROJECT_DIR, TORCH_PYTHON, CLASSICAL_PYTHON, PLAN, SCRIPTS)
